# 623 Stride: source-input-fair LSTM only

Training and inference use the same lossless PC64 + cache-line58 encoder on exactly the external inputs read by `stride.cc`. A compact exact-PC-keyed LSTM learns an unweighted Bernoulli hurdle, conditional Poisson excess, and a three-component direct-delta mixture. Stateless SHA-256 event-keyed inverse-CDF sampling gives every capacity the same event-local quantiles; train/guard history never burns RNG state. Captured Stride requests are labels and comparator replay only; no tracker capacity, degree, cutoff, candidate, page-offset table, same-page rule, or future row enters inference.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID='623_offline_lstm_stride_keyed_crn_v15_seed7'; DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_stride/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified',archive)

In [ ]:
import gzip, json
TRACE='623.xalancbmk_s-700B'; POLICY='stride'; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','candidates':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'} for role in ROLES}
for role,items in INPUTS.items():
    for path in items.values(): assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':'stride_source_input_variable_delta_free_running_v9','neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['pc','addr'],'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'inference_policy_hardcodes_used':False,'nn_generates_own_target_addresses':True}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}; assert not bad,bad
assert manifest['training_runtime_fields']==['pc','addr']==manifest['inference_runtime_fields']
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/train_and_offline_infer.py'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[
 {'tag':'independent_delta_stride_lstm_h8','family':'lstm','size':8,'pair':'p0','parameters':1923},
 {'tag':'independent_delta_stride_lstm_h16','family':'lstm','size':16,'pair':'p1','parameters':5243},
 {'tag':'independent_delta_stride_lstm_h32','family':'lstm','size':32,'pair':'p2','parameters':16107},
 {'tag':'independent_delta_stride_lstm_h64','family':'lstm','size':64,'pair':'p3','parameters':54731},
 {'tag':'independent_delta_stride_lstm_h128','family':'lstm','size':128,'pair':'p4','parameters':199563},
]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
 cmd += ['--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--decoder-seed','7','--epochs','12','--chunk-len','256','--pc-batch-size','128']
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm','model_revision':'compact_pc_keyed_crn_event_sampled_mixture_v15','parameter_count':spec['parameters'],'runtime_feature_count':122,'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'decoder_free_running_self_test':'PASS','normal_policy_outputs_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'probability_threshold_used':False,'neural_degree_cap':None,'common_random_numbers_across_capacities':True,'strict_common_random_numbers_across_capacities':True,'cross_event_rng_state_used':False,'stochastic_decoding_reproducible':True,'cross_event_probability_credit_used':False,'sampled_outputs_used_as_decoder_feedback':False,'decoder_probability_mass_carries_train_guard_history':False,'decoder_sampling_roles':['eval'],'decoder_train_sampling_performed':False,'decoder_guard_sampling_performed':False,'gate_decoding_rule':'event_keyed_bernoulli_inverse_cdf','request_count_training_objective':'unweighted_bernoulli_hurdle_plus_positive_poisson_excess_nll','request_count_decoding_rule':'event_keyed_bernoulli_plus_common_quantile_poisson_inverse_cdf','request_count_residual_scope':'none_event_local','delta_mixture_decoding_rule':'event_keyed_mean_sorted_categorical_inverse_cdf_then_component_mean','delta_decoder_feedback_rule':'complete_mixture_expectation_same_in_training_and_inference','event_keyed_crn_self_test':'PASS','experiment_revision':'stride_source_input_variable_delta_free_running_v9'}
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['training_runtime_fields']==['pc','addr']==meta['inference_runtime_fields']
 encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}; assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
 sampler=meta.get('decoder_sampler',{}); assert sampler.get('sampler_revision')=='sha256_event_keyed_inverse_cdf_crn_v1' and sampler.get('poisson_backend')=='scipy.stats.poisson.ppf' and sampler.get('cross_event_rng_state') is False,sampler
 assert meta.get('decoder_free_running_self_test')=='PASS',meta.get('decoder_free_running_self_test')
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':'stride_source_input_variable_delta_free_running_v9','model_revision':'compact_pc_keyed_crn_event_sampled_mixture_v15','decoder_seed':7,'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the output archive to the matching server run and launch replay. Older threshold/budget/future-window outputs are intentionally rejected by the server metadata checks.